In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import os.path as osp

DS_PATH = "/kaggle/input/ethz-cil-monocular-depth-estimation-2025"

print(os.listdir(DS_PATH))

TRAIN_PATH = osp.join(DS_PATH, "train/train")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

['train_list.txt', 'test_list.txt', 'create_prediction_csv.py', 'test', 'train']


In [2]:

import subprocess
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("git_access_token")

repo_link = "https://" + secret_value_0 + "@github.com/aryansood/CIL.git"

os.chdir("/kaggle/working")

subprocess.run(["rm -rf CIL"], shell=True)
subprocess.run(["git clone -b auto_run " + repo_link], shell=True)

del repo_link

print("repo cloned")
os.chdir("/kaggle/working/CIL")

Cloning into 'CIL'...


repo cloned


In [3]:
import wandb
wandb_key = user_secrets.get_secret("wandb_api_key")

wandb.login(
    key=wandb_key
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 10847706 (10847706-ethz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
!pip install lightning

In [ ]:
from training import begin_training_loop
from models.unet_vit_depth_estimator import UNetViT
import albumentations as A
from pathlib import Path

augmentations = [
    A.Compose([
        A.GaussNoise((0.05,0.05)),
        A.ToTensorV2()
    ]),
    A.Compose([
        A.GaussNoise((0.05, 0.05)),
        A.VerticalFlip(),
        A.ToTensorV2()
    ])
]



begin_training_loop(
    model = UNetViT(learning_rate=0.001, 
                    img_height=426, 
                    img_width=560, 
                    patch_height=16, 
                    patch_width=16),
    data_dir = Path(TRAIN_PATH),
    augmentations = augmentations,
    batch_size=8,
    num_worker=4
)


/usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.5' (you have '2.0.4'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
INFO: Seed set to 80
INFO: Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
INFO: 
  | Name       | Type          | Params | Mode 
-----------------------------------------------------
0 | vit        | ViT           | 39.6 M | train
1 | up1        | UpSampleLayer | 2.1 M  | train
2 | up2        | UpSampleLayer | 525 K  | train
3 | up3        | UpSampleLayer | 131 K  | train
4 | up4        | UpSampleLayer | 33.0 K | train
5 | up         | Sequential    | 2.8 M  | train
6 | final_conv | Conv2d        | 577    | train
-----------------------------------------------------
42.3 M    Trainable params
0         Non-trainable params
42.3 M    Total params
169.373   Total estimated model params size (MB)
130       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 8. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Training: |          | 0/? [00:00<?, ?it/s]